In [ ]:
"""
Topic Classification: TF-IDF + Naive Bayes / SVM
Dataset: AG News (4 classes: World, Sports, Business, Sci/Tech)

Setup:
    pip install datasets scikit-learn pandas --break-system-packages


from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time


In [ ]:
dataset = load_dataset("fancyzhx/ag_news")

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

label_names = dataset["train"].features["label"].names
print(f"Classes: {label_names}")
print(f"Train size: {len(train_texts)}, Test size: {len(test_texts)}")

In [ ]:

print("\nFitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=20000,      # cap vocab size for speed
    ngram_range=(1, 2),      # unigrams + bigrams help short text
    stop_words="english",
    sublinear_tf=True        # log-scale term frequency, standard trick
)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)


In [ ]:

print("\n Naive Bayes ")
start = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train, train_labels)
print(f"Trained in {time.time() - start:.2f}s")

nb_preds = nb_model.predict(X_test)
print(f"Accuracy: {accuracy_score(test_labels, nb_preds):.4f}")
print(classification_report(test_labels, nb_preds, target_names=label_names))

In [ ]:
print("\nLinear SVM ")
start = time.time()
svm_model = LinearSVC(max_iter=2000)
svm_model.fit(X_train, train_labels)
print(f"Trained in {time.time() - start:.2f}s")

svm_preds = svm_model.predict(X_test)
print(f"Accuracy: {accuracy_score(test_labels, svm_preds):.4f}")
print(classification_report(test_labels, svm_preds, target_names=label_names))


In [ ]:
print("\n Random Forest ")
start = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200,     # more trees = better but slower; 200 is a reasonable tradeoff
    max_depth=None,       # let trees grow fully
    n_jobs=-1,            # use all CPU cores
    random_state=42
)
rf_model.fit(X_train, train_labels)
print(f"Trained in {time.time() - start:.2f}s")

rf_preds = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(test_labels, rf_preds):.4f}")
print(classification_report(test_labels, rf_preds, target_names=label_names))


In [ ]:
def predict_topic(text, model=svm_model):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return label_names[pred]

sample_headlines = [
    "Arsenal wins the championship final",
    "Central bank raises interest rates amid inflation fears",
    "New AI chip promises 10x faster processing speeds",
    "UN Security Council to meet over regional conflict"
]

print("\nSample Predictions (comparing all models) ")
for headline in sample_headlines:
    nb_pred = predict_topic(headline, nb_model)
    svm_pred = predict_topic(headline, svm_model)
    rf_pred = predict_topic(headline, rf_model)
    print(f"'{headline}'")
    print(f"   NB: {nb_pred} | SVM: {svm_pred} | RF: {rf_pred}")


print("\n Summary ")
print(f"Naive Bayes    : {accuracy_score(test_labels, nb_preds):.4f}")
print(f"Linear SVM     : {accuracy_score(test_labels, svm_preds):.4f}")
print(f"Random Forest  : {accuracy_score(test_labels, rf_preds):.4f}")
